# 10 – Performance data generator

Performance tuning için büyük tablolar `workspace.perf` schema'sına yazılır: bronze `trans` 100 katına çıkarılır (105,6M satır), biri skew'süz biri skew'lü iki tablo. Diğer katmanlara dokunulmaz.

## 1. Hazırlık: perf schema'sı ve trans profili

Amaç: Performance tuning verisi için `workspace.perf` schema'sını oluşturmak ve çoğaltma kararlarının varsayımlarını kontrol etmek.

Çıktı nasıl okunur:

- `max_trans_id` 10.000.000'dan küçük olmalı; kopyalarda ID'ler `trans_id + copy_no × 10.000.000` ile kaydırıldığı için çakışma olmaz.

- `min_date` / `max_date` 1993–1998 arasında olmalı (Berka formatı `YYMMDD`); her kopya 6 yıl ileri kaydırılınca dönemler üst üste binmez.

- İkinci tablo en çok işlemi olan 5 hesabı gösterir; skew için hot account bunlardan seçilir.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.perf")

df_trans_bronze = spark.table("workspace.bronze.trans")

# Bronze'da her şey string; max'ı sayı olarak bulmak için int'e çevrilir
# Tarihler YYMMDD string ve hepsi 19xx olduğu için string min/max doğru sıralar
df_profile = df_trans_bronze.select(
    F.count("*").alias("rows"),
    F.max(F.col("trans_id").cast("int")).alias("max_trans_id"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.countDistinct("account_id").alias("accounts"),
)
display(df_profile)

df_top_accounts = (
    df_trans_bronze.groupBy("account_id")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)
display(df_top_accounts)

## 2. Generator ve parametreler

`src/perf_data_generator.py` perf tablolarını üç adımda üretir:

| Function | Ne yapar |
|---|---|
| `to_perf_trans` | Bronze `trans`'ı tipli hale getirir (`trans_id`, `account_id` int; `trans_date` date; `amount`, `balance` decimal(12,2)) |
| `multiply_rows` | Satırları `factor` kadar çoğaltır; `trans_id + copy_no × 10.000.000`, tarih `copy_no × 6` yıl ileri |
| `add_skew` | Kopya satırların (`copy_no > 0`) ~%30'unu hot account'a taşır; orijinal satırlara dokunmaz |

`run` bu adımları uygular ve iki tablo yazar: `trans_big` (skew'süz) ve `trans_skewed` (skew'lü).

Parametreler alttaki hücrede; yazma bölüm 3'te.

In [0]:
import os
import sys
import importlib

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import perf_data_generator
importlib.reload(silver_transforms)
importlib.reload(perf_data_generator)

RUN_FACTOR = 100       # 1 orijinal + 99 kopya
HOT_ACCOUNT_ID = 8261  # bronze'da en çok işlemi olan hesap
SKEW_RATIO = 0.3       # hot account'a taşınan kopya satır oranı
SEED = 42

## 3. Tabloları yazma (factor 100)

`run` iki tabloyu `workspace.perf` altına `overwrite` ile yazar (~1 dk).

Neden 100: factor 5'te (26,5 MB) sorgular arasında süre farkı görünmedi; EXPLAIN'de Q2'de shuffle yoktu (bkz. `11_perf_baseline`). Tarih aralığı 1993–2592 (6 yıllık kaydırma korunur; veri sentetik, gelecek tarih kabul edildi).

Nasıl okunur: `rows` 105.632.000, `max_trans_id` 993.682.987, tarih 1993-01-01 – 2592-12-31. `hot_rows`: `trans_big`'de 67.500 (675 × 100), `trans_skewed`'de ~31,4M. `numFiles` ve `size_mb` tabloların fiziksel durumunu gösterir.

In [0]:
perf_data_generator.run(
    spark,
    factor=RUN_FACTOR,
    skew_ratio=SKEW_RATIO,
    hot_account_id=HOT_ACCOUNT_ID,
    seed=SEED,
)

df_details = None
for table_name in ["trans_big", "trans_skewed"]:
    full_name = f"{perf_data_generator.PERF_SCHEMA}.{table_name}"

    # fiziksel durum: dosya sayısı ve boyut
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_name}").select(
        F.lit(table_name).alias("table_name"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
    )

    # içerik kontrolü: satır, hot account, tarih ve ID aralığı
    df_stats = spark.table(full_name).agg(
        F.count("*").alias("rows"),
        F.count_if(F.col("account_id") == HOT_ACCOUNT_ID).alias("hot_rows"),
        F.min("trans_date").alias("min_date"),
        F.max("trans_date").alias("max_date"),
        F.max("trans_id").alias("max_trans_id"),
    )

    # iki tarafta da tek satır var; crossJoin yan yana koyar
    df_detail = df_detail.crossJoin(df_stats)

    if df_details is None:
        df_details = df_detail
    else:
        df_details = df_details.unionByName(df_detail)

display(df_details)